# Ovarian cohort: the model inputs

The two inputs of the CloneVI fit of the ovarian cohort and the cell-type labels of the source study, built from the outputs of the preprocessing (Methods; `preprocess/ovarian/README.md`):

| output | made from |
|---|---|
| `cohort_Ovarian_atac_top25.h5ad`: raw counts of the top 25% of peaks (cell-state input) | `cohort_Ovarian_atac.h5ad` of `build_atac_cohort_signac.R` (Signac, all GSMs merged) |
| `cohort_Ovarian_mtsnv_perdonor_frac0.5pct.h5ad`: mtDNA depth and mutant counts of 64 variants | per-donor variant statistics and clonotypes of `clonotypes_perdonor_signac.R`, mgatk outputs of every library |
| `tableS2_ovarian_tumor_labels.csv`: cell types of the source study | Supplementary Table S2 of the source study (Liu et al.) |

`PRE` is the folder of the preprocessing outputs (shell and R steps, run on the cluster and with Signac). The files are written to `OUT` and compared with the deposited ones (`data/mitotrek/`).

In [1]:
import os
import glob

import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
import anndata as ad

In [2]:
PRE = '../../GSE302113/test_h5ad'                     # outputs of the shell / R preprocessing
DEP = '../../data/mitotrek'                            # deposited model inputs
OUT = '../../data/mitotrek/rebuilt'
os.makedirs(OUT, exist_ok=True)

## 1. Cell-state input: the top 25% of peaks

`build_atac_cohort_signac.R` merged the 11 libraries, kept the cells passing QC and ran `FindTopFeatures(min.cutoff = 'q75')`; its h5ad holds the normalized counts in `X`, the raw counts in `raw.X`, and the percentile of each peak's total count in `var['percentile']`. The model input is the raw counts of the peaks at or above the 75th percentile.

In [3]:
atac = ad.read_h5ad(f'{PRE}/cohort_Ovarian_atac.h5ad')
atac

.../site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


AnnData object with n_obs × n_vars = 108260 × 343278
    obs: 'orig.ident', 'nCount_peaks', 'nFeature_peaks', 'gsm', 'donor', 'tissue', 'sorting', 'cohort', 'passed_filters', 'peak_region_fragments', 'pct_reads_in_peaks', 'blacklist_ratio', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'peaks_snn_res.0.7', 'seurat_clusters', 'lsi_clusters', 'nCount_RNA', 'nFeature_RNA'
    var: 'count', 'percentile'
    uns: 'neighbors'
    obsm: 'X_lsi', 'X_umap'
    varm: 'LSI'
    obsp: 'distances'

QC of the cells (Methods): 1,000–50,000 peak counts, more than 40% of reads in peaks, blacklist ratio below 0.05, nucleosome signal below 4, TSS enrichment above 3. All kept cells satisfy them:

In [4]:
o = atac.obs
qc = pd.Series({'1,000 <= peak counts <= 50,000': o['nCount_peaks'].between(1000, 50000).all(),
                'reads in peaks > 40%': (o['pct_reads_in_peaks'] > 40).all(),
                'blacklist ratio < 0.05': (o['blacklist_ratio'] < 0.05).all(),
                'nucleosome signal < 4': (o['nucleosome_signal'] < 4).all(),
                'TSS enrichment > 3': (o['TSS.enrichment'] > 3).all()})
qc, atac.n_obs, o['donor'].nunique(), o['gsm'].nunique()

(1,000 <= peak counts <= 50,000    True
 reads in peaks > 40%              True
 blacklist ratio < 0.05            True
 nucleosome signal < 4             True
 TSS enrichment > 3                True
 dtype: bool,
 108260,
 5,
 11)

In [5]:
keep = (atac.var['percentile'] >= 0.75).to_numpy()
obs = atac.obs.copy()
for c in obs.columns[obs.dtypes == object]:                 # text columns stored as categories
    obs[c] = obs[c].astype('category')
top25 = ad.AnnData(X=atac.raw.X[:, keep].astype(np.float32), obs=obs, var=atac.var[keep].copy(),
                   obsm={'X_lsi': atac.obsm['X_lsi'], 'X_umap': atac.obsm['X_umap']})
top25.uns['note'] = "cloneVI input: raw counts, top-25% (q75) peaks, float32. obs['donor']=batch(sample_info)."
top25.write_h5ad(f'{OUT}/cohort_Ovarian_atac_top25.h5ad')
top25

AnnData object with n_obs × n_vars = 108260 × 85859
    obs: 'orig.ident', 'nCount_peaks', 'nFeature_peaks', 'gsm', 'donor', 'tissue', 'sorting', 'cohort', 'passed_filters', 'peak_region_fragments', 'pct_reads_in_peaks', 'blacklist_ratio', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'peaks_snn_res.0.7', 'seurat_clusters', 'lsi_clusters', 'nCount_RNA', 'nFeature_RNA'
    var: 'count', 'percentile'
    uns: 'note'
    obsm: 'X_lsi', 'X_umap'

Compared with the deposited input:

In [6]:
dep = ad.read_h5ad(f'{DEP}/cohort_Ovarian_atac_top25.h5ad')
pd.Series({'peaks': (dep.var_names == top25.var_names).all(), 'var': dep.var.equals(top25.var),
           'cells and obs': dep.obs.equals(top25.obs), 'counts': (dep.X != top25.X).nnz == 0,
           'X_lsi': np.array_equal(dep.obsm['X_lsi'], top25.obsm['X_lsi']),
           'X_umap': np.array_equal(dep.obsm['X_umap'], top25.obsm['X_umap'])})

peaks            True
var              True
cells and obs    True
counts           True
X_lsi            True
X_umap           True
dtype: bool

In [7]:
del atac, top25, dep

## 2. mtDNA input: 64 variants called per donor

`clonotypes_perdonor_signac.R` ran the Signac mitochondrial vignette in each donor and wrote the statistics of every candidate variant (`cohort_Ovarian_perdonor_variant_stats.tsv`). A variant is high-confidence in a donor if its strand correlation is at least 0.65, its variance-mean ratio above 0.01, and it is confidently detected in at least $\max(5, \lceil 0.005 N_{\text{donor}} \rceil)$ cells of the donor (Methods); the input is the union over donors.

In [8]:
cells_df = pd.read_csv(f'{PRE}/cohort_Ovarian_atac_cells.csv')       # the cells of the ATAC input, in its order
cells_df['cell'] = cells_df['cell'].astype(str)
cohort_cells = cells_df['cell'].to_numpy()
cell_index = {c: i for i, c in enumerate(cohort_cells)}
ncell_donor = cells_df.groupby('donor')['cell'].size()
samples = list(pd.unique(cells_df['gsm']))
ncell_donor

donor
SU-O-001     8877
SU-O-002    24080
SU-O-003     2085
SU-O-004    35602
SU-O-005    37616
Name: cell, dtype: int64

In [9]:
MIN_STRANDCOR, MIN_VMR, MIN_CELLS_FLOOR, FRAC = 0.65, 0.01, 5, 0.005
vs = pd.read_csv(f'{PRE}/cohort_Ovarian_perdonor_variant_stats.tsv', sep='\t')
vs['position'] = vs['position'].astype(int)
threshold = {d: max(MIN_CELLS_FLOOR, int(np.ceil(FRAC * n))) for d, n in ncell_donor.items()}
hc = vs[(vs['strand_correlation'] >= MIN_STRANDCOR) & (vs['vmr'] > MIN_VMR)
        & (vs['n_cells_conf_detected'] >= vs['donor'].map(threshold))].copy()
variants = sorted(hc['variant'].unique(), key=lambda v: (int(hc.loc[hc['variant'] == v, 'position'].iloc[0]), v))
pd.Series(threshold, name='confident cells required'), len(variants)

(SU-O-001     45
 SU-O-002    121
 SU-O-003     11
 SU-O-004    179
 SU-O-005    189
 Name: confident cells required, dtype: int64,
 64)

For each variant, the donors in which it was called, and the statistics of the donor where its variance-mean ratio is highest:

In [10]:
sel = {}
for v in variants:
    rows = hc[hc['variant'] == v]
    best = rows.loc[rows['vmr'].astype(float).idxmax()]
    sel[v] = dict(donors=sorted(rows['donor'].unique()), position=int(best['position']), nucleotide=str(best['nucleotide']),
                  vmr=float(best['vmr']), strand_correlation=float(best['strand_correlation']),
                  n_cells_conf_detected=int(best['n_cells_conf_detected']), mean_heteroplasmy=float(best['mean']),
                  mean_coverage=float(best['mean_coverage']))
pd.DataFrame(sel).T[['donors', 'n_cells_conf_detected', 'vmr', 'strand_correlation']]

,donors,n_cells_conf_detected,vmr,strand_correlation
146T>C,[SU-O-002],141,0.449958,0.961301
195T>C,[SU-O-001],210,0.915207,0.902361
245T>C,[SU-O-004],270,0.67232,0.964987
310T>C,"[SU-O-002, SU-O-003, SU-O-005]",15283,0.017588,0.829108
515A>G,"[SU-O-001, SU-O-002]",1115,0.037562,0.820353
...,...,...,...,...
15392G>A,[SU-O-004],273,1.117281,0.976267
15601T>C,[SU-O-003],37,0.357697,0.996663
15810C>A,[SU-O-003],28,0.17123,0.946427
16387A>G,[SU-O-003],64,0.029579,0.739495


Depth $d$ and mutant counts $b$ of each variant in each cell, from the mgatk output of each library: $d$ is the coverage at the position and $b$ the forward plus reverse counts of the alternate base. Cells of a library that did not pass the ATAC QC are left out. (This reads about 7.5 GB of mgatk matrices per library.)

In [11]:
nC, nV = len(cohort_cells), len(variants)
vpos = np.array([sel[v]['position'] for v in variants])
valt = np.array([sel[v]['nucleotide'].split('>')[1] for v in variants])
D = np.zeros((nC, nV), dtype=np.float32)
B = np.zeros((nC, nV), dtype=np.float32)
positions = sorted(set(vpos.tolist()))
filled = {}
for gsm in samples:
    mg = f'{PRE}/pack_{gsm}/mgatk'
    genes = pd.read_csv(f'{mg}/genes.tsv', sep='\t')
    gpos = (genes['start'] if 'start' in genes else genes.iloc[:, 1]).astype(int).to_numpy()
    pos_to_row = {p: i for i, p in enumerate(gpos)}
    barcodes = pd.read_csv(f'{mg}/barcodes.tsv', header=None)[0].astype(str).to_numpy()
    crow = np.array([cell_index.get(f'{gsm}#{bc}', -1) for bc in barcodes])
    valid = crow >= 0
    rowpos = [p for p in positions if p in pos_to_row]
    rows = [pos_to_row[p] for p in rowpos]
    k_of = {p: k for k, p in enumerate(rowpos)}
    cov = sio.mmread(f'{mg}/coverage.mtx').tocsr()[rows, :].toarray()
    alt = {a: (sio.mmread(glob.glob(f'{mg}/{a}_counts_fw.mtx')[0]).tocsr()
               + sio.mmread(glob.glob(f'{mg}/{a}_counts_rev.mtx')[0]).tocsr())[rows, :].toarray() for a in sorted(set(valt))}
    for j in range(nV):
        if vpos[j] in k_of:
            D[crow[valid], j] = cov[k_of[vpos[j]], valid]
            B[crow[valid], j] = alt[valt[j]][k_of[vpos[j]], valid]
    filled[gsm] = (int(valid.sum()), len(barcodes))
pd.DataFrame(filled, index=['cells filled', 'mgatk cells']).T

,cells filled,mgatk cells
GSM9096478,8877,11019
GSM9096479,8523,13671
GSM9096480,15557,20002
GSM9096481,2085,2488
GSM9096482,7626,8876
GSM9096483,11686,15936
GSM9096484,16290,20002
GSM9096485,10361,13397
GSM9096486,5705,8258
GSM9096487,9892,13473


In [12]:
clo = pd.read_csv(f'{PRE}/cohort_Ovarian_clonotypes.csv').set_index('cell')
obs_mg = cells_df.set_index('cell').loc[cohort_cells, ['gsm', 'donor', 'tissue']].copy()
obs_mg['clonotype_signac'] = clo['clonotype'].reindex(cohort_cells).to_numpy()
obs_mg['mtDNA_depth'] = clo['mtDNA_depth'].reindex(cohort_cells).to_numpy().astype(np.float32)
var = pd.DataFrame(index=pd.Index(variants))
var['selected_in_donors'] = [','.join(sel[v]['donors']) for v in variants]
var['n_donors_selected'] = [len(sel[v]['donors']) for v in variants]
for c in ['position', 'nucleotide', 'vmr', 'strand_correlation', 'n_cells_conf_detected', 'mean_heteroplasmy', 'mean_coverage']:
    var[f'vs_{c}'] = [sel[v][c] for v in variants]
mtsnv = ad.AnnData(X=D.copy(), obs=obs_mg, var=var)
mtsnv.layers['all_mut'] = B.copy()
with np.errstate(divide='ignore', invalid='ignore'):
    mtsnv.layers['VAF'] = np.nan_to_num(B / D).astype(np.float32)
mtsnv.uns['cohort'] = 'Ovarian'
mtsnv.uns['variant_selection'] = dict(method='per_donor_union', selection='frac0.5pct', spec=dict(frac=FRAC),
                                      min_strandcor=MIN_STRANDCOR, min_vmr=MIN_VMR, cells_floor=MIN_CELLS_FLOOR,
                                      source='Signac mito vignette IdentifyVariants per donor')
mtsnv.write(f'{OUT}/cohort_Ovarian_mtsnv_perdonor_frac0.5pct.h5ad', compression='gzip')
mtsnv

AnnData object with n_obs × n_vars = 108260 × 64
    obs: 'gsm', 'donor', 'tissue', 'clonotype_signac', 'mtDNA_depth'
    var: 'selected_in_donors', 'n_donors_selected', 'vs_position', 'vs_nucleotide', 'vs_vmr', 'vs_strand_correlation', 'vs_n_cells_conf_detected', 'vs_mean_heteroplasmy', 'vs_mean_coverage'
    uns: 'cohort', 'variant_selection'
    layers: 'all_mut', 'VAF'

Compared with the deposited input:

In [13]:
dep = ad.read_h5ad(f'{DEP}/cohort_Ovarian_mtsnv_perdonor_frac0.5pct.h5ad')
pd.Series({'variants': (dep.var_names == mtsnv.var_names).all(), 'var': dep.var.equals(mtsnv.var),
           'cells and obs': dep.obs.equals(mtsnv.obs), 'depth': np.array_equal(dep.X, mtsnv.X),
           'mutant counts': np.array_equal(dep.layers['all_mut'], mtsnv.layers['all_mut']),
           'VAF': np.array_equal(dep.layers['VAF'], mtsnv.layers['VAF'])})

variants         True
var              True
cells and obs    True
depth            True
mutant counts    True
VAF              True
dtype: bool

## 3. Cell types of the source study

The 'Ovarian_Tumor' sheet of Supplementary Table S2 of the source study gives a cell type for the tumor cells of each of its samples. Each sample is matched to a GEO library (GSM) by the overlap of cell barcodes, and the labels are indexed by the cell names of the CloneVI input (`<GSM>#<barcode>`).

In [14]:
s2 = pd.read_excel(f'{DEP}/paper_supp/NIHMS2184724-supplement-Table_S2.xlsx', 'Ovarian_Tumor', header=1)
s2['sample'] = s2['sample'].astype(str)
s2['bc'] = s2['cell_index'].str.split('#').str[1]
ours = pd.DataFrame({'gsm': cells_df['gsm'].astype(str).to_numpy(), 'bc': cells_df['cell'].str.split('#').str[1].to_numpy()},
                    index=cohort_cells)
rows = []
for s, g in s2.groupby('sample'):
    ov = {gsm: len(set(g['bc']) & set(ours['bc'][ours['gsm'] == gsm])) for gsm in ours['gsm'].unique()}
    best = max(ov, key=ov.get)
    rows.append((s, best, ov[best], sorted(ov.values())[-2], len(g)))
matching = pd.DataFrame(rows, columns=['sample', 'gsm', 'shared barcodes', 'next best', 'cells in Table S2'])
matching

,sample,gsm,shared barcodes,next best,cells in Table S2
0,672,GSM9096478,7711,211,8192
1,679,GSM9096479,6896,220,9180
2,687-D2,GSM9096481,2019,56,2198
3,690,GSM9096482,6719,181,7649
4,691,GSM9096483,9436,257,10817
5,693,GSM9096485,7777,181,8091
6,694,GSM9096486,5120,141,6027


In [15]:
s2['obs_name'] = s2['sample'].map(dict(zip(matching['sample'], matching['gsm']))) + '#' + s2['bc']
KEEP = ['obs_name', 'sample', 'patient', 'sort', 'cluster', 'cell_type_lv0', 'cell_type_lv1', 'cell_type_lv2', 'nFrags',
        'TSSEnrichment', 'clone']
labels = s2[KEEP].set_index('obs_name')
labels.to_csv(f'{OUT}/tableS2_ovarian_tumor_labels.csv')
labelled = labels.index.isin(cohort_cells)
pd.Series({'Table S2 cells found in the input': f'{labelled.sum():,} / {len(labels):,}',
           'libraries with labels': labels.index[labelled].str.split('#').str[0].nunique(),
           'libraries': ours['gsm'].nunique()})

Table S2 cells found in the input    45,678 / 52,154
libraries with labels                              7
libraries                                         11
dtype: object

In [16]:
dep_labels = pd.read_csv(f'{DEP}/paper_supp/tableS2_ovarian_tumor_labels.csv', index_col=0)
dep_labels.equals(pd.read_csv(f'{OUT}/tableS2_ovarian_tumor_labels.csv', index_col=0))

True

## 4. The numbers in the paper (Methods l.328)

In [17]:
rows = [('cells', '108,260', f'{len(cohort_cells):,}'), ('donors', '5', str(ncell_donor.size)),
        ('libraries', '11', str(len(samples))), ('peaks of the cell-state input', '85,859', f'{int(keep.sum()):,}'),
        ('variants in the union', '64', str(nV)),
        ('labelled cells', '45,678', f'{labelled.sum():,}'),
        ('libraries with labels', '7 of 11', f"{labels.index[labelled].str.split('#').str[0].nunique()} of {len(samples)}")]
numbers = pd.DataFrame(rows, columns=['item', 'paper', 'computed'])
numbers['match'] = numbers['paper'] == numbers['computed']
numbers

,item,paper,computed,match
0,cells,"108,260","108,260",True
1,donors,5,5,True
2,libraries,11,11,True
3,peaks of the cell-state input,"85,859","85,859",True
4,variants in the union,64,64,True
5,labelled cells,"45,678","45,678",True
6,libraries with labels,7 of 11,7 of 11,True
